# NeoOLAF × RAGTree — FULL parallel EventStoryLine → FinCausal

This is the full benchmark runner built from the **process-isolated parallel architecture that passed the 5-document validation**.

## Full dataset order

1. **EventStoryLine** — 443 normalized records — frozen `v1.7`
2. **FinCausal** — 967 normalized records — frozen `unified-v1.3.1-selection-hotfix`

FinCausal is not started until EventStoryLine is fully complete.

## Parallelism

### EventStoryLine
- **5 independent document processes at a time**
- **4 layer workers inside each process**
- maximum intended document-level fan-out: 5
- maximum intended layer-level fan-out: 4 per document

### FinCausal
- **5 independent document processes at a time**
- **1 layer worker inside each process**

The controller keeps the process pool full: whenever one document finishes, the next pending document is launched, up to 5 concurrent documents.

## Scientific / budget guarantees

- Frozen development configs are reused unchanged.
- Development/smoke manifest is **read-only**.
- Gold fields are stripped before Layer 0.
- Gold is written/evaluated only after Layer 12 returns.
- Every document has its own run directory.
- EventStoryLine gets a separate cache namespace per document process.
- Child stdout/stderr logs live outside document run directories, avoiding the Windows file-handle issue found in the first process-isolated attempt.
- Successful documents are persisted immediately and skipped on restart.
- Failed documents are **not retried during the same invocation**; rerunning the notebook retries only unresolved records.
- After 3 failures in one invocation, the controller stops launching new documents to protect API budget, lets already-running processes finish, saves state, and aborts.
- Exact UTC usage sessions are persisted for later OpenRouter token/cost accounting.

Use a fresh kernel and run from the top.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, time, subprocess, textwrap, re, traceback, uuid as _uuid
from pprint import pprint

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def find_project_root():
    candidates = []
    env = os.environ.get("NEOOLAF_PROJECT_ROOT")
    if env:
        candidates.append(Path(env))
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    candidates.append(Path(r"C:\Users\galencarmedeiro\NeoOLAF"))
    for p in candidates:
        if (p / "src" / "neoolaf").is_dir() and (p / "examples").is_dir():
            return p.resolve()
    raise FileNotFoundError("NeoOLAF project root not found. Set NEOOLAF_PROJECT_ROOT.")

PROJECT_ROOT = find_project_root()
EXPERIMENT_ROOT = PROJECT_ROOT / "examples" / "RAGTreeDatasets"
TOOLS_DIR = EXPERIMENT_ROOT / "tools"

for p in [PROJECT_ROOT, PROJECT_ROOT / "src", TOOLS_DIR]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import ragtree_experiment_state_v1 as expstate

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Python executable for child processes:", sys.executable)


PROJECT_ROOT: C:\Users\galencarmedeiro\NeoOLAF
Python executable for child processes: c:\Users\galencarmedeiro\NeoOLAF\.venv\Scripts\python.exe


## Run controls

`RUN_PAID=True` is intentional. Change it to `False` only if you want to run the zero-cost preflight without launching the benchmark.


In [2]:
RUN_PAID = True
RUN_ORDER = ["eventstoryline", "fincausal"]

MODEL_NAME = "openai/gpt-oss-20b"
OPENROUTER_HOST = "https://openrouter.ai/api/v1"
REASONING_EFFORT = "minimal"
MAX_TOKENS = 8192
REQUEST_TIMEOUT = 180

DOCUMENT_WORKERS = {
    "eventstoryline": 5,
    "fincausal": 5,
}

LAYER_WORKERS = {
    "eventstoryline": 4,
    "fincausal": 1,
}

EXPECTED_VERSIONS = {
    "eventstoryline": "v1.7",
    "fincausal": "unified-v1.3.1-selection-hotfix",
}

EXPECTED_RECORD_COUNTS = {
    "eventstoryline": 443,
    "fincausal": 967,
}

# Budget / fault guard.
MAX_FAILURES_PER_INVOCATION = 3

# How often to refresh consolidated summaries while a dataset is running.
SUMMARY_CHECKPOINT_EVERY = 10

assert RUN_ORDER == ["eventstoryline", "fincausal"]
assert DOCUMENT_WORKERS == {"eventstoryline": 5, "fincausal": 5}
assert LAYER_WORKERS == {"eventstoryline": 4, "fincausal": 1}
assert MAX_FAILURES_PER_INVOCATION >= 1

print("RUN_PAID:", RUN_PAID)
print("MODEL:", MODEL_NAME)
print("EventStoryLine: 5 document processes × 4 layer workers")
print("FinCausal     : 5 document processes × 1 layer worker")


RUN_PAID: True
MODEL: openai/gpt-oss-20b
EventStoryLine: 5 document processes × 4 layer workers
FinCausal     : 5 document processes × 1 layer worker


## Zero-cost full-dataset preflight

This verifies the exact dataset sizes, frozen versions, completed smoke gates, unique record keys, configs, ontologies, and gold isolation across **all 1,410 records**.

No API calls are made here.


In [3]:
RAGTREE_ROOT = expstate.discover_ragtree_root(PROJECT_ROOT)
PREPROCESSED_DIR = expstate.discover_preprocessed_dir(RAGTREE_ROOT)
ONTOLOGY_ROOT = expstate.discover_ontology_dir(RAGTREE_ROOT)

DATASET_FILES = expstate.locate_dataset_files(PREPROCESSED_DIR)
RAW_ONTOLOGY_FILES = expstate.locate_ontology_files(ONTOLOGY_ROOT)

ONTOLOGY_FILES = {
    "eventstoryline": RAW_ONTOLOGY_FILES["eventstoryline"],
    "fincausal": RAW_ONTOLOGY_FILES["fincausal"],
}

CONFIGS = {
    "eventstoryline": {
        "profile": EXPERIMENT_ROOT / "configs/eventstoryline_profile_native_ablation_v1_7.json",
        "guidance": EXPERIMENT_ROOT / "configs/guidance_eventstoryline_native_ablation_v1_7.json",
        "task": EXPERIMENT_ROOT / "configs/eventstoryline_task_guidance_v1_7.json",
        "catalog": EXPERIMENT_ROOT / "ontology/eventstoryline_relation_catalog.json",
        "aliases": EXPERIMENT_ROOT / "ontology/eventstoryline_relation_aliases.json",
        "version": "v1.7",
    },
    "fincausal": {
        "profile": EXPERIMENT_ROOT / "configs/fincausal_profile_unified_v1_3.json",
        "guidance": EXPERIMENT_ROOT / "configs/fincausal_guidance_unified_v1_3.json",
        "task": EXPERIMENT_ROOT / "configs/fincausal_task_guidance_unified_v1_3.json",
        "catalog": EXPERIMENT_ROOT / "ontology/fincausal_relation_catalog.json",
        "aliases": EXPERIMENT_ROOT / "ontology/fincausal_relation_aliases.json",
        "version": "unified-v1.3.1-selection-hotfix",
    },
}

for k, cfg in CONFIGS.items():
    assert cfg["version"] == EXPECTED_VERSIONS[k], (k, cfg["version"])
    for name, p in cfg.items():
        if name != "version":
            assert Path(p).exists(), (k, name, p)

for k, p in ONTOLOGY_FILES.items():
    assert Path(p).exists(), (k, p)

dataset_rows = {
    k: expstate.read_jsonl(DATASET_FILES[k])
    for k in RUN_ORDER
}

for k in RUN_ORDER:
    assert len(dataset_rows[k]) == EXPECTED_RECORD_COUNTS[k], (
        k, len(dataset_rows[k]), EXPECTED_RECORD_COUNTS[k]
    )
    keys = [expstate.record_key(k, r) for r in dataset_rows[k]]
    assert len(keys) == len(set(keys)), f"{k}: record_key collision in full dataset"

STATE_DIR = EXPERIMENT_ROOT / "state"
TEMPLATE_MANIFEST = STATE_DIR / "development_manifest_TEMPLATE_v1.json"
LIVE_MANIFEST = STATE_DIR / "development_manifest_v1.json"
dev_manifest = expstate.load_manifest(LIVE_MANIFEST, TEMPLATE_MANIFEST)

for k in RUN_ORDER:
    e = dev_manifest[k]
    assert e.get("one_doc_completed"), f"{k}: one-doc development gate incomplete"
    assert e.get("smoke5_already_run"), f"{k}: fixed smoke-5 incomplete"
    assert e.get("best_version") == EXPECTED_VERSIONS[k], (
        k, e.get("best_version"), EXPECTED_VERSIONS[k]
    )

# Full anti-leak audit, every record.
for k in RUN_ORDER:
    for r in dataset_rows[k]:
        clean = expstate.strip_gold(r)
        forbidden = {"entities", "relations", "pred_relations", "ontology_links"} & set(clean)
        assert not forbidden, (k, expstate.record_key(k, r), forbidden)

# Basic dataset audits.
def audit_dataset(k, rows):
    rel_counts = {}
    gold_entity_counts = []
    positive_relation_docs = 0
    for r in rows:
        gold_entity_counts.append(len(r.get("entities") or {}))
        doc_rel_count = 0
        for rel, pairs in (r.get("relations") or {}).items():
            n = len(pairs or [])
            rel_counts[rel] = rel_counts.get(rel, 0) + n
            if str(rel).lower() not in {"null", "none", ""}:
                doc_rel_count += n
        if doc_rel_count > 0:
            positive_relation_docs += 1
    return {
        "dataset": k,
        "records": len(rows),
        "positive_relation_docs": positive_relation_docs,
        "relations": rel_counts,
        "mean_gold_entities": (
            sum(gold_entity_counts) / len(gold_entity_counts)
            if gold_entity_counts else 0.0
        ),
    }

audit = {k: audit_dataset(k, dataset_rows[k]) for k in RUN_ORDER}

observed_esl = {
    str(x).upper()
    for x in audit["eventstoryline"]["relations"]
    if str(x).lower() not in {"null", "none", ""}
}
observed_fc = {
    str(x).upper()
    for x in audit["fincausal"]["relations"]
    if str(x).lower() not in {"null", "none", ""}
}
assert observed_esl == {"PRECONDITION", "FALLING_ACTION"}, observed_esl
assert observed_fc == {"CAUSE"}, observed_fc

print("RAGTREE_ROOT:", RAGTREE_ROOT)
print("PREPROCESSED_DIR:", PREPROCESSED_DIR)
print("\nFull-dataset audit:")
pprint(audit)
print("\nFrozen version/smoke gate: OK")
print("Unique full-dataset record keys: OK")
print("Gold isolation across all 1,410 records: OK")
print("No API call has been made by this cell.")


RAGTREE_ROOT: C:\Users\galencarmedeiro\RAGTree
PREPROCESSED_DIR: C:\Users\galencarmedeiro\RAGTree\preprocessed

Full-dataset audit:
{'eventstoryline': {'dataset': 'eventstoryline',
                    'mean_gold_entities': 11.753950338600452,
                    'positive_relation_docs': 443,
                    'records': 443,
                    'relations': {'FALLING_ACTION': 4880,
                                  'PRECONDITION': 4760,
                                  'null': 55}},
 'fincausal': {'dataset': 'fincausal',
               'mean_gold_entities': 1.9493278179937952,
               'positive_relation_docs': 929,
               'records': 967,
               'relations': {'CAUSE': 929}}}

Frozen version/smoke gate: OK
Unique full-dataset record keys: OK
Gold isolation across all 1,410 records: OK
No API call has been made by this cell.


## Full-run state and isolated worker script

The full experiment uses a new run root:

`runs/full_process_isolated_eventstoryline_fincausal_v1`

The worker is the same process-isolation design validated on the parallel 5-document test:

- separate Python interpreter per document;
- separate document run directory;
- separate EventStoryLine cache namespace;
- console logs stored outside run directories;
- post-L12 evaluation only.


In [4]:
RUNS_ROOT = EXPERIMENT_ROOT / "runs" / "full_process_isolated_eventstoryline_fincausal_v1"
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

WORKER_SCRIPT = RUNS_ROOT / "_full_worker.py"
PROGRESS_PATH = RUNS_ROOT / "full_progress.json"
SUMMARY_PATH = RUNS_ROOT / "full_summary_live.json"
USAGE_SESSIONS_PATH = RUNS_ROOT / "openrouter_usage_sessions.json"
CONSOLE_LOG_ROOT = RUNS_ROOT / "_worker_console_logs"
CONSOLE_LOG_ROOT.mkdir(parents=True, exist_ok=True)

worker_code = r"""
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
import argparse, os, sys, json, time, shutil, re, traceback

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def safe_dir_name(text):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text))[:120]

parser = argparse.ArgumentParser()
parser.add_argument("--project-root", required=True)
parser.add_argument("--dataset", required=True, choices=["eventstoryline", "fincausal"])
parser.add_argument("--record-key", required=True)
parser.add_argument("--run-root", required=True)
parser.add_argument("--layer-workers", required=True, type=int)
parser.add_argument("--model", required=True)
parser.add_argument("--host", required=True)
parser.add_argument("--reasoning-effort", default="minimal")
parser.add_argument("--max-tokens", type=int, default=8192)
parser.add_argument("--request-timeout", type=int, default=180)
args = parser.parse_args()

PROJECT_ROOT = Path(args.project_root).resolve()
EXPERIMENT_ROOT = PROJECT_ROOT / "examples" / "RAGTreeDatasets"
TOOLS_DIR = EXPERIMENT_ROOT / "tools"

for p in [PROJECT_ROOT, PROJECT_ROOT / "src", TOOLS_DIR]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import ragtree_experiment_state_v1 as expstate

dataset_key = args.dataset
rkey = args.record_key
run_root = Path(args.run_root).resolve()
run_dir = run_root / dataset_key / "full" / safe_dir_name(rkey)

# Critical for EventStoryLine process isolation:
# each document gets its own cache namespace.
if dataset_key == "eventstoryline":
    os.environ["NEOOLAF_EVENTSTORYLINE_CACHE_DIR"] = str(
        run_root / "_isolated_esl_cache" / safe_dir_name(rkey)
    )

import ragtree_dataset_adapters_v1_8 as adapters
import eventstoryline_native_ablation_v1_7 as esl_v17

failure_path = run_dir / "worker_failure.json"

try:
    RAGTREE_ROOT = expstate.discover_ragtree_root(PROJECT_ROOT)
    PREPROCESSED_DIR = expstate.discover_preprocessed_dir(RAGTREE_ROOT)
    ONTOLOGY_ROOT = expstate.discover_ontology_dir(RAGTREE_ROOT)
    DATASET_FILES = expstate.locate_dataset_files(PREPROCESSED_DIR)
    RAW_ONTOLOGY_FILES = expstate.locate_ontology_files(ONTOLOGY_ROOT)

    rows = expstate.read_jsonl(DATASET_FILES[dataset_key])
    matches = [r for r in rows if expstate.record_key(dataset_key, r) == rkey]
    if len(matches) != 1:
        raise RuntimeError(
            f"{dataset_key} {rkey}: expected exactly one normalized row, got {len(matches)}"
        )
    gold_record = matches[0]

    configs = {
        "eventstoryline": {
            "profile": EXPERIMENT_ROOT / "configs/eventstoryline_profile_native_ablation_v1_7.json",
            "guidance": EXPERIMENT_ROOT / "configs/guidance_eventstoryline_native_ablation_v1_7.json",
            "task": EXPERIMENT_ROOT / "configs/eventstoryline_task_guidance_v1_7.json",
            "catalog": EXPERIMENT_ROOT / "ontology/eventstoryline_relation_catalog.json",
            "aliases": EXPERIMENT_ROOT / "ontology/eventstoryline_relation_aliases.json",
            "version": "v1.7",
        },
        "fincausal": {
            "profile": EXPERIMENT_ROOT / "configs/fincausal_profile_unified_v1_3.json",
            "guidance": EXPERIMENT_ROOT / "configs/fincausal_guidance_unified_v1_3.json",
            "task": EXPERIMENT_ROOT / "configs/fincausal_task_guidance_unified_v1_3.json",
            "catalog": EXPERIMENT_ROOT / "ontology/fincausal_relation_catalog.json",
            "aliases": EXPERIMENT_ROOT / "ontology/fincausal_relation_aliases.json",
            "version": "unified-v1.3.1-selection-hotfix",
        },
    }
    cfg = configs[dataset_key]
    ontology_path = RAW_ONTOLOGY_FILES[dataset_key]

    # A failed/incomplete record may leave a partial directory. Clean only this record.
    if run_dir.exists():
        shutil.rmtree(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)

    pre_gold_contract = expstate.gold_contract_summary(dataset_key, gold_record)

    clean_record = expstate.strip_gold(gold_record)
    forbidden = {"entities", "relations", "pred_relations", "ontology_links"} & set(clean_record)
    if forbidden:
        raise RuntimeError(f"Gold leakage in worker input: {forbidden}")

    input_path = run_dir / "pipeline_input_NO_GOLD.jsonl"
    expstate.write_jsonl(input_path, [clean_record])

    gold_path = run_dir / "POSTHOC_GOLD_AFTER_LAYER12.jsonl"
    if gold_path.exists():
        gold_path.unlink()

    api_key = os.environ.get("OPENROUTER_API_KEY", "").strip().strip('"').strip("'")
    if not api_key:
        raise RuntimeError("OPENROUTER_API_KEY is missing.")

    started = utc_now()
    t0 = time.perf_counter()

    if dataset_key == "eventstoryline":
        final_state = esl_v17.run_native_pipeline(
            project_root=PROJECT_ROOT,
            input_jsonl=input_path,
            ontology_path=ontology_path,
            profile_path=cfg["profile"],
            guidance_path=cfg["guidance"],
            task_guidance_path=cfg["task"],
            relation_catalog_path=cfg["catalog"],
            relation_aliases_path=cfg["aliases"],
            run_dir=run_dir,
            model_name=args.model,
            api_key=api_key,
            host=args.host,
            workers=args.layer_workers,
            max_tokens=args.max_tokens,
            request_timeout=args.request_timeout,
            reasoning_effort=args.reasoning_effort,
            verbose=True,
            clean_run_dir=False,
        )

        # Gold becomes available only AFTER native Layer 12 returned.
        expstate.write_jsonl(
            gold_path,
            [{k: v for k, v in gold_record.items() if not k.startswith("__")}],
        )
        summary = esl_v17.analyze_run(
            run_dir=run_dir,
            gold_jsonl=gold_path,
            catalog_path=cfg["catalog"],
            aliases_path=cfg["aliases"],
        )

        result = {
            "dataset": dataset_key,
            "version": cfg["version"],
            "record_key": rkey,
            "document_id": gold_record.get("document_id"),
            "title": gold_record.get("title"),
            "relation_metrics": (
                summary.get("projected_relation_evaluation")
                or summary.get("strict_relation_evaluation")
                or {}
            ),
            "endpoint_metrics": (
                summary.get("relation_endpoint_evaluation")
                or summary.get("event_entity_evaluation")
                or {}
            ),
            "candidate_pool": (
                summary.get("candidate_pool_coverage")
                or summary.get("candidate_pool")
                or {}
            ),
            "pre_run_gold_contract": pre_gold_contract,
        }

    else:
        final_state = adapters.run_native_pipeline_record(
            dataset_key=dataset_key,
            project_root=PROJECT_ROOT,
            input_jsonl=input_path,
            ontology_path=ontology_path,
            profile_path=cfg["profile"],
            guidance_path=cfg["guidance"],
            task_guidance_path=cfg["task"],
            relation_catalog_path=cfg["catalog"],
            relation_aliases_path=cfg["aliases"],
            run_dir=run_dir,
            model_name=args.model,
            api_key=api_key,
            host=args.host,
            workers=args.layer_workers,
            max_tokens=args.max_tokens,
            request_timeout=args.request_timeout,
            reasoning_effort=args.reasoning_effort,
            verbose=True,
            clean_run_dir=False,
        )

        # Gold becomes available only AFTER native Layer 12 returned.
        expstate.write_jsonl(
            gold_path,
            [{k: v for k, v in gold_record.items() if not k.startswith("__")}],
        )
        result = adapters.evaluate_state(dataset_key, final_state, gold_record)
        result.update({
            "dataset": dataset_key,
            "version": cfg["version"],
            "record_key": rkey,
            "document_id": gold_record.get("document_id"),
            "title": gold_record.get("title"),
            "pre_run_gold_contract": pre_gold_contract,
        })

        # On positive FinCausal records, protect against an evaluator/projection bug.
        expected_gold = int(pre_gold_contract["gold_target_relation_count"])
        evaluated_gold = int((result.get("relation_metrics") or {}).get("gold", 0) or 0)
        if expected_gold > 0 and evaluated_gold == 0:
            raise RuntimeError(
                f"FinCausal evaluator integrity error: controller expected "
                f"{expected_gold} gold CAUSE relation(s), evaluator saw 0."
            )

    result.update({
        "run_dir": str(run_dir),
        "started_at_utc": started,
        "finished_at_utc": utc_now(),
        "elapsed_seconds": time.perf_counter() - t0,
        "model": args.model,
        "layer_workers": args.layer_workers,
        "process_id": os.getpid(),
        "gold_visible_to_pipeline": False,
    })

    adapters.write_json(run_dir / "posthoc_evaluation.json", result)
    print(
        "WORKER_SUCCESS",
        dataset_key,
        rkey,
        json.dumps(result.get("relation_metrics") or {}),
    )
    sys.exit(0)

except Exception as exc:
    try:
        run_dir.mkdir(parents=True, exist_ok=True)
        payload = {
            "dataset": dataset_key,
            "record_key": rkey,
            "error_type": type(exc).__name__,
            "error": str(exc),
            "traceback": traceback.format_exc(),
            "failed_at_utc": utc_now(),
            "process_id": os.getpid(),
        }
        failure_path.write_text(
            json.dumps(payload, indent=2, ensure_ascii=False),
            encoding="utf-8",
        )
    except Exception:
        pass
    traceback.print_exc()
    sys.exit(1)
"""

WORKER_SCRIPT.write_text(textwrap.dedent(worker_code), encoding="utf-8")
compile(WORKER_SCRIPT.read_text(encoding="utf-8"), str(WORKER_SCRIPT), "exec")

def atomic_json(path, obj):
    expstate.atomic_write_json(Path(path), obj)

def load_json(path, default=None):
    p = Path(path)
    if not p.exists():
        return default
    return json.loads(p.read_text(encoding="utf-8"))

def dataset_record_keys(dataset_key):
    return [expstate.record_key(dataset_key, r) for r in dataset_rows[dataset_key]]

def fresh_progress():
    return {
        "schema_version": 1,
        "experiment": "full_process_isolated_eventstoryline_fincausal_v1",
        "created_at_utc": utc_now(),
        "model": MODEL_NAME,
        "host": OPENROUTER_HOST,
        "reasoning_effort": REASONING_EFFORT,
        "max_tokens": MAX_TOKENS,
        "dataset_order": list(RUN_ORDER),
        "datasets": {
            k: {
                "version": EXPECTED_VERSIONS[k],
                "document_workers": DOCUMENT_WORKERS[k],
                "layer_workers": LAYER_WORKERS[k],
                "total_records": len(dataset_rows[k]),
                "record_keys_sha": __import__("hashlib").sha256(
                    "\n".join(dataset_record_keys(k)).encode("utf-8")
                ).hexdigest(),
                "completed_record_keys": [],
                "failures": {},
                "first_started_at_utc": None,
                "fully_finished_at_utc": None,
            }
            for k in RUN_ORDER
        },
        "usage_sessions": [],
    }

progress = load_json(PROGRESS_PATH, None)
if progress is None:
    progress = fresh_progress()
    atomic_json(PROGRESS_PATH, progress)
else:
    # Hard resume guards: refuse silent experiment drift.
    assert progress["model"] == MODEL_NAME
    assert progress["host"] == OPENROUTER_HOST
    assert progress["dataset_order"] == RUN_ORDER
    for k in RUN_ORDER:
        ds = progress["datasets"][k]
        assert ds["version"] == EXPECTED_VERSIONS[k]
        assert ds["document_workers"] == DOCUMENT_WORKERS[k]
        assert ds["layer_workers"] == LAYER_WORKERS[k]
        assert ds["total_records"] == len(dataset_rows[k])
        current_sha = __import__("hashlib").sha256(
            "\n".join(dataset_record_keys(k)).encode("utf-8")
        ).hexdigest()
        assert ds["record_keys_sha"] == current_sha, (
            k, "full dataset record-key sequence changed"
        )

def write_usage_sessions():
    payload = {
        "experiment": progress["experiment"],
        "model": MODEL_NAME,
        "host": OPENROUTER_HOST,
        "usage_sessions": progress.get("usage_sessions") or [],
        "note": (
            "Use these UTC windows with model=openai/gpt-oss-20b in OpenRouter. "
            "Multiple sessions are kept so interrupted/resumed runs do not create one misleading giant time window."
        ),
    }
    atomic_json(USAGE_SESSIONS_PATH, payload)
    return payload

write_usage_sessions()

print("RUNS_ROOT:", RUNS_ROOT)
print("Worker script:", WORKER_SCRIPT)
print("Worker syntax check: OK")
print("Console logs:", CONSOLE_LOG_ROOT)
for k in RUN_ORDER:
    ds = progress["datasets"][k]
    print(
        f"{k:15s}: {len(ds['completed_record_keys'])}/{ds['total_records']} completed "
        f"| workers={ds['document_workers']}×{ds['layer_workers']}"
    )


RUNS_ROOT: C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\runs\full_process_isolated_eventstoryline_fincausal_v1
Worker script: C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\runs\full_process_isolated_eventstoryline_fincausal_v1\_full_worker.py
Worker syntax check: OK
Console logs: C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\runs\full_process_isolated_eventstoryline_fincausal_v1\_worker_console_logs
eventstoryline : 0/443 completed | workers=5×4
fincausal      : 0/967 completed | workers=5×1


## Aggregation and consolidated results

The aggregate is schema-tolerant across the EventStoryLine and FinCausal evaluators.

Primary relation reporting:
- micro precision / recall / F1;
- TP / FP / FN;
- macro document F1 (diagnostic);
- macro F1 restricted to positive-gold documents.

Endpoint metrics are also aggregated when available.

A consolidated JSONL is written per dataset from completed per-document evaluations.


In [5]:
def safe_dir_name(text):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text))[:120]

def result_path(dataset_key, rkey):
    return (
        RUNS_ROOT
        / dataset_key
        / "full"
        / safe_dir_name(rkey)
        / "posthoc_evaluation.json"
    )

def _metric_count(m, *names):
    if not isinstance(m, dict):
        return 0
    for name in names:
        if name in m and m[name] is not None:
            return int(m[name] or 0)
    return 0

def normalize_counts(m):
    tp = _metric_count(m, "tp", "true_positive")
    fp = _metric_count(m, "fp", "false_positive")
    fn = _metric_count(m, "fn", "false_negative")
    pred = _metric_count(m, "pred", "predicted")
    gold = _metric_count(m, "gold", "gold_unique")

    if pred == 0 and (tp + fp) > 0:
        pred = tp + fp
    if gold == 0 and (tp + fn) > 0:
        gold = tp + fn

    return {"pred": pred, "gold": gold, "tp": tp, "fp": fp, "fn": fn}

def prf(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) else 0.0
    return p, r, f1

def load_completed_results(dataset_key):
    completed = set(progress["datasets"][dataset_key].get("completed_record_keys") or [])
    rows = []
    missing = []
    for source_row in dataset_rows[dataset_key]:
        rkey = expstate.record_key(dataset_key, source_row)
        if rkey not in completed:
            continue
        p = result_path(dataset_key, rkey)
        if not p.exists():
            missing.append((rkey, str(p)))
            continue
        rows.append(json.loads(p.read_text(encoding="utf-8")))
    if missing:
        raise RuntimeError(
            f"{dataset_key}: progress says completed but result file missing: {missing[:3]}"
        )
    return rows

def aggregate(dataset_key):
    rows = load_completed_results(dataset_key)

    rel = [normalize_counts(r.get("relation_metrics") or {}) for r in rows]
    ep = [normalize_counts(r.get("endpoint_metrics") or {}) for r in rows]

    rel_tp = sum(x["tp"] for x in rel)
    rel_fp = sum(x["fp"] for x in rel)
    rel_fn = sum(x["fn"] for x in rel)
    rel_pred = sum(x["pred"] for x in rel)
    rel_gold = sum(x["gold"] for x in rel)
    rel_p, rel_r, rel_f1 = prf(rel_tp, rel_fp, rel_fn)

    ep_tp = sum(x["tp"] for x in ep)
    ep_fp = sum(x["fp"] for x in ep)
    ep_fn = sum(x["fn"] for x in ep)
    ep_pred = sum(x["pred"] for x in ep)
    ep_gold = sum(x["gold"] for x in ep)
    ep_p, ep_r, ep_f1 = prf(ep_tp, ep_fp, ep_fn)

    doc_f1s = [
        float((r.get("relation_metrics") or {}).get("f1", 0.0) or 0.0)
        for r in rows
    ]
    positive_doc_f1s = [
        float((r.get("relation_metrics") or {}).get("f1", 0.0) or 0.0)
        for r, c in zip(rows, rel)
        if c["gold"] > 0
    ]

    ds = progress["datasets"][dataset_key]
    completed_n = len(ds.get("completed_record_keys") or [])
    total = ds["total_records"]

    return {
        "dataset": dataset_key,
        "version": EXPECTED_VERSIONS[dataset_key],
        "status": "COMPLETE" if completed_n == total else "INCOMPLETE",
        "completed_records": completed_n,
        "total_records": total,
        "pending_records": total - completed_n,
        "recorded_failures": len(ds.get("failures") or {}),
        "document_processes": DOCUMENT_WORKERS[dataset_key],
        "layer_workers_per_process": LAYER_WORKERS[dataset_key],
        "relation": {
            "pred": rel_pred,
            "gold": rel_gold,
            "tp": rel_tp,
            "fp": rel_fp,
            "fn": rel_fn,
            "precision": rel_p,
            "recall": rel_r,
            "micro_f1": rel_f1,
            "macro_doc_f1": (
                sum(doc_f1s) / len(doc_f1s)
                if doc_f1s else 0.0
            ),
            "macro_positive_gold_doc_f1": (
                sum(positive_doc_f1s) / len(positive_doc_f1s)
                if positive_doc_f1s else 0.0
            ),
            "positive_gold_docs_completed": len(positive_doc_f1s),
        },
        "endpoint": {
            "pred": ep_pred,
            "gold": ep_gold,
            "tp": ep_tp,
            "fp": ep_fp,
            "fn": ep_fn,
            "precision": ep_p,
            "recall": ep_r,
            "micro_f1": ep_f1,
        },
        "first_started_at_utc": ds.get("first_started_at_utc"),
        "fully_finished_at_utc": ds.get("fully_finished_at_utc"),
    }

def export_consolidated(dataset_key):
    rows = load_completed_results(dataset_key)
    path = RUNS_ROOT / f"{dataset_key}_completed_results.jsonl"
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    tmp.replace(path)
    return path

def save_live_summary():
    summary = {
        "experiment": progress["experiment"],
        "model": MODEL_NAME,
        "generated_at_utc": utc_now(),
        "datasets": {k: aggregate(k) for k in RUN_ORDER},
    }
    atomic_json(SUMMARY_PATH, summary)
    write_usage_sessions()
    return summary

print("Aggregation helpers ready. No API calls made.")


Aggregation helpers ready. No API calls made.


## Execute full benchmark

This is the paid cell.

The controller uses a **bounded process pool**:
- starts up to 5 pending documents;
- polls them;
- persists each success immediately;
- launches the next pending document whenever a slot opens;
- never exceeds 5 simultaneous document processes.

If you interrupt the notebook, rerun from the top later. Completed records are skipped.


In [6]:
def start_usage_session():
    session = {
        "session_id": _uuid.uuid4().hex,
        "started_at_utc": utc_now(),
        "finished_at_utc": None,
        "datasets": {},
    }
    progress.setdefault("usage_sessions", []).append(session)
    atomic_json(PROGRESS_PATH, progress)
    write_usage_sessions()
    return session

def finish_usage_session(session):
    if session.get("finished_at_utc") is None:
        session["finished_at_utc"] = utc_now()
    atomic_json(PROGRESS_PATH, progress)
    write_usage_sessions()

def start_dataset_window(session, dataset_key):
    session["datasets"][dataset_key] = {
        "started_at_utc": utc_now(),
        "finished_at_utc": None,
        "document_workers": DOCUMENT_WORKERS[dataset_key],
        "layer_workers": LAYER_WORKERS[dataset_key],
        "completed_at_start": len(progress["datasets"][dataset_key]["completed_record_keys"]),
        "completed_at_end": None,
    }
    if progress["datasets"][dataset_key].get("first_started_at_utc") is None:
        progress["datasets"][dataset_key]["first_started_at_utc"] = session["datasets"][dataset_key]["started_at_utc"]
    atomic_json(PROGRESS_PATH, progress)
    write_usage_sessions()

def finish_dataset_window(session, dataset_key):
    win = session["datasets"].get(dataset_key)
    if win and win.get("finished_at_utc") is None:
        win["finished_at_utc"] = utc_now()
        win["completed_at_end"] = len(progress["datasets"][dataset_key]["completed_record_keys"])
    atomic_json(PROGRESS_PATH, progress)
    write_usage_sessions()

def launch_child(dataset_key, gold_record):
    rkey = expstate.record_key(dataset_key, gold_record)

    log_dir = CONSOLE_LOG_ROOT / dataset_key
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / f"{safe_dir_name(rkey)}.log"
    log_handle = open(log_path, "w", encoding="utf-8", buffering=1)

    cmd = [
        sys.executable,
        str(WORKER_SCRIPT),
        "--project-root", str(PROJECT_ROOT),
        "--dataset", dataset_key,
        "--record-key", rkey,
        "--run-root", str(RUNS_ROOT),
        "--layer-workers", str(LAYER_WORKERS[dataset_key]),
        "--model", MODEL_NAME,
        "--host", OPENROUTER_HOST,
        "--reasoning-effort", REASONING_EFFORT,
        "--max-tokens", str(MAX_TOKENS),
        "--request-timeout", str(REQUEST_TIMEOUT),
    ]

    env = os.environ.copy()
    env["NEOOLAF_PROJECT_ROOT"] = str(PROJECT_ROOT)

    proc = subprocess.Popen(
        cmd,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        env=env,
        cwd=str(PROJECT_ROOT),
    )

    return {
        "proc": proc,
        "record_key": rkey,
        "document_id": gold_record.get("document_id"),
        "title": gold_record.get("title"),
        "log_handle": log_handle,
        "log_path": log_path,
        "started_monotonic": time.monotonic(),
    }

def run_full_dataset(dataset_key):
    ds = progress["datasets"][dataset_key]
    total = ds["total_records"]
    completed = set(ds.get("completed_record_keys") or [])

    pending_rows = [
        r for r in dataset_rows[dataset_key]
        if expstate.record_key(dataset_key, r) not in completed
    ]

    if not pending_rows:
        print(f"\nSKIP {dataset_key}: already complete {total}/{total}.")
        if ds.get("fully_finished_at_utc") is None:
            ds["fully_finished_at_utc"] = utc_now()
            atomic_json(PROGRESS_PATH, progress)
        return True

    print(
        f"\n=== FULL {dataset_key.upper()} ===\n"
        f"total={total} | completed={len(completed)} | pending={len(pending_rows)}\n"
        f"document_processes={DOCUMENT_WORKERS[dataset_key]} | "
        f"layer_workers/process={LAYER_WORKERS[dataset_key]}"
    )

    queue = iter(pending_rows)
    active = {}
    failures_this_invocation = 0
    successes_this_invocation = 0
    stop_launching = False
    last_status_print = 0.0

    def launch_next():
        nonlocal stop_launching
        if stop_launching:
            return False
        try:
            row = next(queue)
        except StopIteration:
            return False
        child = launch_child(dataset_key, row)
        active[child["record_key"]] = child
        print(
            f"START pid={child['proc'].pid} | {child['record_key']} | "
            f"overall={len(ds['completed_record_keys'])}/{total}"
        )
        return True

    for _ in range(DOCUMENT_WORKERS[dataset_key]):
        if not launch_next():
            break

    try:
        while active:
            now = time.monotonic()
            finished_keys = []

            for rkey, info in list(active.items()):
                rc = info["proc"].poll()
                if rc is None:
                    continue

                info["log_handle"].close()
                finished_keys.append(rkey)
                wall = time.monotonic() - info["started_monotonic"]

                if rc == 0 and result_path(dataset_key, rkey).exists():
                    if rkey not in ds["completed_record_keys"]:
                        ds["completed_record_keys"].append(rkey)
                    ds.setdefault("failures", {}).pop(rkey, None)
                    successes_this_invocation += 1

                    print(
                        f"DONE pid={info['proc'].pid} | {rkey} | "
                        f"wall={wall:.1f}s | "
                        f"completed={len(ds['completed_record_keys'])}/{total}"
                    )

                    atomic_json(PROGRESS_PATH, progress)

                    if (
                        len(ds["completed_record_keys"]) % SUMMARY_CHECKPOINT_EVERY == 0
                        or len(ds["completed_record_keys"]) == total
                    ):
                        live = save_live_summary()
                        a = live["datasets"][dataset_key]
                        print(
                            f"CHECKPOINT {dataset_key}: "
                            f"{a['completed_records']}/{a['total_records']} | "
                            f"relation micro-F1={a['relation']['micro_f1']:.6f}"
                        )

                else:
                    failures_this_invocation += 1
                    failure_file = (
                        RUNS_ROOT
                        / dataset_key
                        / "full"
                        / safe_dir_name(rkey)
                        / "worker_failure.json"
                    )
                    failure = {
                        "record_key": rkey,
                        "document_id": info.get("document_id"),
                        "title": info.get("title"),
                        "returncode": rc,
                        "log_path": str(info["log_path"]),
                        "failure_file": (
                            str(failure_file) if failure_file.exists() else None
                        ),
                        "failed_at_utc": utc_now(),
                    }
                    if failure_file.exists():
                        try:
                            failure["worker_failure"] = json.loads(
                                failure_file.read_text(encoding="utf-8")
                            )
                        except Exception:
                            pass

                    ds.setdefault("failures", {})[rkey] = failure
                    atomic_json(PROGRESS_PATH, progress)
                    save_live_summary()

                    print(
                        f"FAILED pid={info['proc'].pid} | {rkey} | rc={rc} | "
                        f"failures_this_invocation="
                        f"{failures_this_invocation}/{MAX_FAILURES_PER_INVOCATION}\n"
                        f"  log: {info['log_path']}"
                    )

                    if failures_this_invocation >= MAX_FAILURES_PER_INVOCATION:
                        stop_launching = True
                        print(
                            "FAILURE BUDGET REACHED: no new documents will be launched. "
                            "Already-running processes are allowed to finish."
                        )

            for rkey in finished_keys:
                active.pop(rkey, None)

            # Refill free process slots unless fault guard stopped us.
            while (
                not stop_launching
                and len(active) < DOCUMENT_WORKERS[dataset_key]
            ):
                if not launch_next():
                    break

            if active and now - last_status_print >= 30:
                print(
                    f"STATUS {dataset_key}: "
                    f"completed={len(ds['completed_record_keys'])}/{total} | "
                    f"active={len(active)} | "
                    f"failures_this_invocation={failures_this_invocation}"
                )
                last_status_print = now

            if active:
                time.sleep(1.0)

    except KeyboardInterrupt:
        print("\nKeyboardInterrupt: terminating active child processes...")
        for info in active.values():
            try:
                info["proc"].terminate()
            except Exception:
                pass
        time.sleep(1.0)
        for info in active.values():
            try:
                if info["proc"].poll() is None:
                    info["proc"].kill()
            except Exception:
                pass
            try:
                info["log_handle"].close()
            except Exception:
                pass
        atomic_json(PROGRESS_PATH, progress)
        save_live_summary()
        raise

    completed_now = len(ds["completed_record_keys"])
    if completed_now == total:
        ds["fully_finished_at_utc"] = utc_now()
        atomic_json(PROGRESS_PATH, progress)
        export_consolidated(dataset_key)
        save_live_summary()
        print(f"\n{dataset_key}: FULL COMPLETE {completed_now}/{total}")
        pprint(aggregate(dataset_key))
        return True

    # Incomplete: normally because failure budget stopped launching more work.
    atomic_json(PROGRESS_PATH, progress)
    export_consolidated(dataset_key)
    save_live_summary()
    print(
        f"\n{dataset_key}: INCOMPLETE {completed_now}/{total}. "
        f"Recorded unresolved failures={len(ds.get('failures') or {})}."
    )
    return False

if not RUN_PAID:
    print("RUN_PAID=False -> stopped before all API calls.")
else:
    api_key = os.environ.get("OPENROUTER_API_KEY", "").strip().strip('"').strip("'")
    if not api_key:
        raise RuntimeError("OPENROUTER_API_KEY is missing.")

    session = start_usage_session()

    try:
        # Dataset 1: EventStoryLine
        start_dataset_window(session, "eventstoryline")
        esl_complete = run_full_dataset("eventstoryline")
        finish_dataset_window(session, "eventstoryline")

        if not esl_complete:
            raise RuntimeError(
                "EventStoryLine full run is incomplete. "
                "FinCausal is intentionally blocked. "
                "Inspect recorded failure logs, then rerun; successful documents will be skipped."
            )

        # Dataset 2: FinCausal, strictly after full ESL completion.
        assert (
            len(progress["datasets"]["eventstoryline"]["completed_record_keys"])
            == progress["datasets"]["eventstoryline"]["total_records"]
        )

        start_dataset_window(session, "fincausal")
        fc_complete = run_full_dataset("fincausal")
        finish_dataset_window(session, "fincausal")

        if not fc_complete:
            raise RuntimeError(
                "FinCausal full run is incomplete. "
                "Inspect recorded failure logs, then rerun; successful documents will be skipped."
            )

        final_summary = save_live_summary()
        print("\nFULL TWO-DATASET PARALLEL EXPERIMENT COMPLETE")
        pprint(final_summary)

    finally:
        # Close any currently open dataset usage windows and the session itself.
        for k in RUN_ORDER:
            if k in session.get("datasets", {}):
                finish_dataset_window(session, k)
        finish_usage_session(session)

    print("\nProgress:", PROGRESS_PATH)
    print("Summary :", SUMMARY_PATH)
    print("Usage sessions:", USAGE_SESSIONS_PATH)



=== FULL EVENTSTORYLINE ===
total=443 | completed=0 | pending=443
document_processes=5 | layer_workers/process=4
START pid=10404 | eventstoryline:0:9476bcab3bb12b52 | overall=0/443
START pid=15556 | eventstoryline:1:e262639faff6efc3 | overall=0/443
START pid=27992 | eventstoryline:2:ff2644df5186a9c2 | overall=0/443
START pid=26164 | eventstoryline:3:529aeeecab91ee4b | overall=0/443
START pid=13916 | eventstoryline:4:a6e7f686d2b2cc24 | overall=0/443
STATUS eventstoryline: completed=0/443 | active=5 | failures_this_invocation=0
STATUS eventstoryline: completed=0/443 | active=5 | failures_this_invocation=0
STATUS eventstoryline: completed=0/443 | active=5 | failures_this_invocation=0
DONE pid=10404 | eventstoryline:0:9476bcab3bb12b52 | wall=88.2s | completed=1/443
START pid=29784 | eventstoryline:5:1e4ee13ec9bd1fdc | overall=1/443
STATUS eventstoryline: completed=1/443 | active=5 | failures_this_invocation=0
DONE pid=15556 | eventstoryline:1:e262639faff6efc3 | wall=98.2s | completed=2/44

RuntimeError: FinCausal full run is incomplete. Inspect recorded failure logs, then rerun; successful documents will be skipped.

## Final / current report — zero API calls

This cell can be run after completion, after an interruption, or after a resumed run.

For OpenRouter accounting, use `openrouter_usage_sessions.json`: it preserves each actual notebook execution window instead of merging interruptions into one large interval.


In [ ]:
summary = save_live_summary()
usage = write_usage_sessions()

print("FULL PARALLEL BENCHMARK STATUS")
print("==============================")

for k in RUN_ORDER:
    a = summary["datasets"][k]
    rel = a["relation"]
    ep = a["endpoint"]

    print(
        f"\n{k} [{a['version']}] "
        f"{a['completed_records']}/{a['total_records']} | {a['status']}"
    )
    print(
        " relation:",
        f"P={rel['precision']:.6f}",
        f"R={rel['recall']:.6f}",
        f"micro-F1={rel['micro_f1']:.6f}",
        f"macro-positive-doc-F1={rel['macro_positive_gold_doc_f1']:.6f}",
        f"TP={rel['tp']}",
        f"FP={rel['fp']}",
        f"FN={rel['fn']}",
    )
    print(
        " endpoint:",
        f"P={ep['precision']:.6f}",
        f"R={ep['recall']:.6f}",
        f"micro-F1={ep['micro_f1']:.6f}",
    )
    print(
        " pending/failures:",
        a["pending_records"],
        "/",
        a["recorded_failures"],
    )

print("\nOPENROUTER USAGE SESSIONS")
pprint(usage)

print("\nFiles to send me after the run:")
print(" - executed notebook")
print(" -", SUMMARY_PATH)
print(" -", USAGE_SESSIONS_PATH)
print(" -", RUNS_ROOT / "eventstoryline_completed_results.jsonl")
print(" -", RUNS_ROOT / "fincausal_completed_results.jsonl")
